<a href="https://colab.research.google.com/github/valsson-group/UNT-ChemicalApplicationsOfMachineLearning-Spring2026/blob/main/Lecture-12_March-3-2026/Lecture-12_Regression-1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Lecture 12 - Regression



In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

Here we create an example dataset for regression

In [ ]:
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split


n_samples=1000

features, target = make_regression(
    n_samples=n_samples,
    n_features=20,
    n_informative=8,
    shuffle=True,
    noise=2.0
)

feature_names = [f"feature {i}" for i in range(features.shape[1])]


In [ ]:
plt.hist(target,bins=60,density=True,label='Discrete Histogram')
sns.kdeplot(target,label='KDE')
plt.title("Target")
plt.xlabel("Target")
plt.show()

for i in range(features.shape[1]):
  plt.hist(features[i],bins=20,density=True,label='Discrete Histogram')
  sns.kdeplot(features[i],label='KDE')
  plt.title(feature_names[i])
  plt.xlabel(feature_names[i])
  plt.show()





In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn.metrics import PredictionErrorDisplay

features_train, features_test, target_train, target_test = train_test_split(features, target, test_size=0.2)

model = RandomForestRegressor()

model.fit(features_train,target_train)

target_train_predicted = model.predict(features_train)
target_test_predicted = model.predict(features_test)

plt.plot(target_test_predicted, target_test,'.',label='Test set')
plt.plot(target_train_predicted, target_train,'.',label='Train set')
diagonal_line =np.linspace(np.min(target),np.max(target),1000)
plt.plot(diagonal_line,diagonal_line, color='red', linestyle='--')
# plt.axline([0, 0], slope=1, color='red', linestyle='--')
plt.legend()
plt.xlabel("Predicted Value")
plt.ylabel("Actual Value")
plt.show()

plt.plot(target_train_predicted, target_train-target_train_predicted,'.',label='Train set')
plt.plot(target_test_predicted, target_test-target_test_predicted,'.',label='Test set')
diagonal_line =np.linspace(np.min(target),np.max(target),1000)
plt.plot(diagonal_line, np.zeros(diagonal_line.size), color='red', linestyle='--')
# plt.axline([0, 0], slope=0, color='red', linestyle='--')
plt.legend()
plt.xlabel("Predicted Value")
plt.ylabel("Residuals (Actual-Predicted)")
plt.show()



display = PredictionErrorDisplay.from_predictions(y_true=target_test, y_pred=target_test_predicted,kind="actual_vs_predicted")
display.plot()

Feature importance

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn.inspection import permutation_importance

features_train, features_test, target_train, target_test = train_test_split(features, target, test_size=0.2)

model = RandomForestRegressor()

model.fit(features_train,target_train)

result = permutation_importance(
    model, features_test, target_test, n_repeats=10, random_state=42, n_jobs=2
)

feature_importances = pd.Series(result.importances_mean, index=feature_names)

fig, ax = plt.subplots()
feature_importances.plot.bar(yerr=result.importances_std, ax=ax)
ax.set_title("Feature importances using permutation on full model")
ax.set_ylabel("Mean accuracy decrease")
fig.tight_layout()
plt.show()



Let's do cross validation

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn.model_selection import cross_validate,ShuffleSplit


model = RandomForestRegressor()

target_used = target.copy()

scoring =['neg_mean_absolute_error',
          'neg_mean_squared_error',
          'neg_root_mean_squared_error',
          'neg_max_error',
          'r2']

# employ 5-fold CV
scores_fold = cross_validate(
    model,
    features, target_used,
    scoring=scoring,
    cv=5,
    return_train_score=True,
    return_estimator=True,
    return_indices=True
)

NumSplits=100
cv_random = ShuffleSplit(n_splits=NumSplits, test_size=0.20)
scores_random = cross_validate(
    model,
    features, target_used,
    scoring=scoring,
    cv=cv_random,
    return_train_score=True,
    return_estimator=True,
    return_indices=True
)

for s in scoring:
  print("Score: {:s}".format(s))
  print("- Training set: {:s}".format(s))
  print("  - 5-Fold CV                   : {:.3f} +- {:.3f}".format( np.nanmean(scores_fold['train_'+s]),np.nanstd(scores_fold['train_'+s])))
  print("  - Random Splits ({:d} splits) : {:.3f} +- {:.3f}".format(NumSplits, np.nanmean(scores_random['train_'+s]), np.nanstd(scores_random['train_'+s])))
  print("- Test set: {:s}".format(s))
  print("  - 5-Fold CV                   : {:.3f} +- {:.3f}".format( np.nanmean(scores_fold['test_'+s]),np.nanstd(scores_fold['test_'+s])))
  print("  - Random Splits ({:d} splits) : {:.3f} +- {:.3f}".format(NumSplits, np.nanmean(scores_random['test_'+s]), np.nanstd(scores_random['test_'+s])))
  print("")



Let's use log10 transformed target values

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn.model_selection import cross_validate,ShuffleSplit


model = RandomForestRegressor()

target_used = np.log10(target)

scoring =['neg_mean_absolute_error',
          'neg_mean_squared_error',
          'neg_root_mean_squared_error',
          'neg_max_error',
          'r2']

# employ 5-fold CV
scores_fold = cross_validate(
    model,
    features, target_used,
    scoring=scoring,
    cv=5,
    return_train_score=True,
    return_estimator=True,
    return_indices=True
)

NumSplits=100
cv_random = ShuffleSplit(n_splits=NumSplits, test_size=0.2)
scores_random = cross_validate(
    model,
    features, target_used,
    scoring=scoring,
    cv=cv_random,
    return_train_score=True,
    return_estimator=True,
    return_indices=True
)

for s in scoring:
  print("Score: {:s}".format(s))
  print("- Training set: {:s}".format(s))
  print("  - 5-Fold CV                   : {:.3f} +- {:.3f}".format( np.nanmean(scores_fold['train_'+s]),np.nanstd(scores_fold['train_'+s])))
  print("  - Random Splits ({:d} splits) : {:.3f} +- {:.3f}".format(NumSplits, np.nanmean(scores_random['train_'+s]), np.nanstd(scores_random['train_'+s])))
  print("- Test set: {:s}".format(s))
  print("  - 5-Fold CV                   : {:.3f} +- {:.3f}".format( np.nanmean(scores_fold['test_'+s]),np.nanstd(scores_fold['test_'+s])))
  print("  - Random Splits ({:d} splits) : {:.3f} +- {:.3f}".format(NumSplits, np.nanmean(scores_random['test_'+s]), np.nanstd(scores_random['test_'+s])))
  print("")



Let's do kNN regression

We can also use some of the example dataset available in sklearn

In [ ]:
from sklearn.datasets import load_diabetes

features, target = load_diabetes(return_X_y=True)

feature_names = [f"feature {i}" for i in range(features.shape[1])]


In [ ]:
plt.hist(target,bins=60,density=True,label='Discrete Histogram')
sns.kdeplot(target,label='KDE')
plt.title("Target")
plt.xlabel("Target")
plt.show()

for i in range(features.shape[1]):
  plt.hist(features[i],bins=20,density=True,label='Discrete Histogram')
  sns.kdeplot(features[i],label='KDE')
  plt.title(feature_names[i])
  plt.xlabel(feature_names[i])
  plt.show()

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn.metrics import PredictionErrorDisplay

features_train, features_test, target_train, target_test = train_test_split(features, target, test_size=0.2)

model = RandomForestRegressor()

model.fit(features_train,target_train)

target_train_predicted = model.predict(features_train)
target_test_predicted = model.predict(features_test)

plt.plot(target_test_predicted, target_test,'.',label='Test set')
plt.plot(target_train_predicted, target_train,'.',label='Train set')
diagonal_line =np.linspace(np.min(target),np.max(target),1000)
plt.plot(diagonal_line,diagonal_line, color='red', linestyle='--')
# plt.axline([0, 0], slope=1, color='red', linestyle='--')
plt.legend()
plt.xlabel("Predicted Value")
plt.ylabel("Actual Value")
plt.show()

plt.plot(target_train_predicted, target_train-target_train_predicted,'.',label='Train set')
plt.plot(target_test_predicted, target_test-target_test_predicted,'.',label='Test set')
diagonal_line =np.linspace(np.min(target),np.max(target),1000)
plt.plot(diagonal_line, np.zeros(diagonal_line.size), color='red', linestyle='--')
# plt.axline([0, 0], slope=0, color='red', linestyle='--')
plt.legend()
plt.xlabel("Predicted Value")
plt.ylabel("Residuals (Actual-Predicted)")
plt.show()



display = PredictionErrorDisplay.from_predictions(y_true=target_test, y_pred=target_test_predicted,kind="actual_vs_predicted")
display.plot()

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn.inspection import permutation_importance

features_train, features_test, target_train, target_test = train_test_split(features, target, test_size=0.2)

model = RandomForestRegressor()

model.fit(features_train,target_train)

result = permutation_importance(
    model, features_test, target_test, n_repeats=10, random_state=42, n_jobs=2
)

feature_importances = pd.Series(result.importances_mean, index=feature_names)

fig, ax = plt.subplots()
feature_importances.plot.bar(yerr=result.importances_std, ax=ax)
ax.set_title("Feature importances using permutation on full model")
ax.set_ylabel("Mean accuracy decrease")
fig.tight_layout()
plt.show()

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn.model_selection import cross_validate,ShuffleSplit


model = RandomForestRegressor()

target_used = target.copy()

scoring =['neg_mean_absolute_error',
          'neg_mean_squared_error',
          'neg_root_mean_squared_error',
          'neg_max_error',
          'r2']

# employ 5-fold CV
scores_fold = cross_validate(
    model,
    features, target_used,
    scoring=scoring,
    cv=5,
    return_train_score=True,
    return_estimator=True,
    return_indices=True
)

NumSplits=100
cv_random = ShuffleSplit(n_splits=NumSplits, test_size=0.20)
scores_random = cross_validate(
    model,
    features, target_used,
    scoring=scoring,
    cv=cv_random,
    return_train_score=True,
    return_estimator=True,
    return_indices=True
)

for s in scoring:
  print("Score: {:s}".format(s))
  print("- Training set: {:s}".format(s))
  print("  - 5-Fold CV                   : {:.3f} +- {:.3f}".format( np.nanmean(scores_fold['train_'+s]),np.nanstd(scores_fold['train_'+s])))
  print("  - Random Splits ({:d} splits) : {:.3f} +- {:.3f}".format(NumSplits, np.nanmean(scores_random['train_'+s]), np.nanstd(scores_random['train_'+s])))
  print("- Test set: {:s}".format(s))
  print("  - 5-Fold CV                   : {:.3f} +- {:.3f}".format( np.nanmean(scores_fold['test_'+s]),np.nanstd(scores_fold['test_'+s])))
  print("  - Random Splits ({:d} splits) : {:.3f} +- {:.3f}".format(NumSplits, np.nanmean(scores_random['test_'+s]), np.nanstd(scores_random['test_'+s])))
  print("")

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn.model_selection import cross_validate,ShuffleSplit


model = RandomForestRegressor()

target_used = np.log10(target)

scoring =['neg_mean_absolute_error',
          'neg_mean_squared_error',
          'neg_root_mean_squared_error',
          'neg_max_error',
          'r2']

# employ 5-fold CV
scores_fold = cross_validate(
    model,
    features, target_used,
    scoring=scoring,
    cv=5,
    return_train_score=True,
    return_estimator=True,
    return_indices=True
)

NumSplits=100
cv_random = ShuffleSplit(n_splits=NumSplits, test_size=0.2)
scores_random = cross_validate(
    model,
    features, target_used,
    scoring=scoring,
    cv=cv_random,
    return_train_score=True,
    return_estimator=True,
    return_indices=True
)

for s in scoring:
  print("Score: {:s}".format(s))
  print("- Training set: {:s}".format(s))
  print("  - 5-Fold CV                   : {:.3f} +- {:.3f}".format( np.nanmean(scores_fold['train_'+s]),np.nanstd(scores_fold['train_'+s])))
  print("  - Random Splits ({:d} splits) : {:.3f} +- {:.3f}".format(NumSplits, np.nanmean(scores_random['train_'+s]), np.nanstd(scores_random['train_'+s])))
  print("- Test set: {:s}".format(s))
  print("  - 5-Fold CV                   : {:.3f} +- {:.3f}".format( np.nanmean(scores_fold['test_'+s]),np.nanstd(scores_fold['test_'+s])))
  print("  - Random Splits ({:d} splits) : {:.3f} +- {:.3f}".format(NumSplits, np.nanmean(scores_random['test_'+s]), np.nanstd(scores_random['test_'+s])))
  print("")

